# Name Hierarchy Levels — SynBio Papers (Mid & High)

Assigns globally unique, publication-ready names to the mid-level and high-level
topic groups for **SynBio Papers**. For each group the LLM sees the low-level
sub-topics (name + description) it contains, then returns one name per group via
OpenAI function calling. Prompts come from `prompts_hierarchy.yaml`.

> Run `get_topic_hierarchy.ipynb` (this folder) **first**.

**Updates** `assets/reports/papers_topic_name_hierarchy.tsv` with `mid_name`
and `high_name` columns.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 04-topic_hierarchy/, where the aux/
# package resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from aux.paths import REPORTS_DIR, OPENAI_MODEL, set_seed
from aux.naming import (
    load_prompts, make_client, build_system_prompt,
    name_hierarchy_level, load_naming_inputs, save_named_hierarchy,
)

set_seed()

# ── CONFIG: SynBio Papers ──────────────────────────────────────────────────
PREFIX = "papers"

prompts = load_prompts()
client = make_client()
system_prompt = build_system_prompt(prompts)

topic_names, hierarchy = load_naming_inputs(PREFIX)
print(f"{PREFIX}: {len(topic_names)} low-level topics | "
      f"mid groups {hierarchy[hierarchy['mid'] >= 0]['mid'].nunique()} | "
      f"high groups {hierarchy[hierarchy['high'] >= 0]['high'].nunique()}")

papers: 225 low-level topics | mid groups 14 | high groups 7


## 1. Name the mid- and high-level groups

In [3]:
mid_names = name_hierarchy_level(
    hierarchy, topic_names, level_col="mid", label="mid",
    client=client, system_prompt=system_prompt, model=OPENAI_MODEL,
)
high_names = name_hierarchy_level(
    hierarchy, topic_names, level_col="high", label="high",
    client=client, system_prompt=system_prompt, model=OPENAI_MODEL,
)

  Naming 14 mid groups via gpt-4.1-nano …
  ✓ 14 mid names assigned
  Naming 7 high groups via gpt-4.1-nano …
  ✓ 7 high names assigned


## 2. Add the name columns and save

In [4]:
hierarchy["mid_name"] = hierarchy["mid"].map(mid_names)
hierarchy["high_name"] = hierarchy["high"].map(high_names)
save_named_hierarchy(hierarchy, PREFIX)

print(f"Saved → {REPORTS_DIR / f'{PREFIX}_topic_name_hierarchy.tsv'}")
hierarchy.head(10)

Saved → /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports/papers_topic_name_hierarchy.tsv


,global_name,low,mid,high,mid_name,high_name
0,Plant Genetic Engineering Tools,0,0,0,Plant Synthetic Biology and Natural Product En...,Plant Synthetic Biology and Natural Product En...
1,Ethics and Society in Synthetic Biology,1,1,1,"Synthetic Biology in Society, Ethics, and Educ...","Synthetic Biology in Society, Ethics, and Educ..."
2,Plant Biosynthesis Pathway Engineering,2,0,0,Plant Synthetic Biology and Natural Product En...,Plant Synthetic Biology and Natural Product En...
3,Synthetic Biology for Diagnostics and Therapeu...,3,2,2,Microbial and Cellular Engineering for Bioprod...,Microbial and Cellular Engineering for Biotech...
4,CRISPR-Cas Systems in Synthetic Biology,4,2,2,Microbial and Cellular Engineering for Bioprod...,Microbial and Cellular Engineering for Biotech...
5,Nucleic Acid Recognition and Innate Immunity,5,3,3,Synthetic Biology in Medicine and Immunology,"Synthetic Biology in Medicine, Vaccines, and P..."
6,DNA Nanotechnology and Nanoscale Construction,6,4,4,DNA Nanotechnology and Nucleic Acid Engineering,DNA Nanotechnology and Structural Synthetic Bi...
7,RNA Regulatory Circuits,7,2,2,Microbial and Cellular Engineering for Bioprod...,Microbial and Cellular Engineering for Biotech...
8,Synthetic Biological Oscillators,8,5,2,Synthetic Biology for Biological Circuits and ...,Microbial and Cellular Engineering for Biotech...
9,Genetic System Design and Optimization,9,2,2,Microbial and Cellular Engineering for Bioprod...,Microbial and Cellular Engineering for Biotech...


## 3. Summary

In [5]:
n_mid = hierarchy["mid_name"].notna().sum()
n_high = hierarchy["high_name"].notna().sum()
print(f"{PREFIX}: {n_mid} topics with mid_name ({hierarchy['mid_name'].dropna().nunique()} unique), "
      f"{n_high} with high_name ({hierarchy['high_name'].dropna().nunique()} unique)")

papers: 225 topics with mid_name (14 unique), 225 with high_name (7 unique)
